# TAAF ARC-AGI2 Kaggle Run

This notebook is generated by the Tufa ARC-AGI Framework (TAAF), an open-source deployment harness from [Tufa Labs](https://tufalabs.ai/) for running ARC-AGI2 solvers reproducibly on Kaggle.

The notebook makes the bundled TAAF source snapshot importable, runs any solver setup commands, loads the pickled benchmark, and writes results to `/kaggle/working`. It can run as a public/offline debug notebook or as the same code path used for competition reruns. Kaggle's `KAGGLE_IS_COMPETITION_RERUN` flag always wins and switches the run into submission mode.

For quick inline experiments, use the customization code cell just before the benchmark run. That is the safest place to tweak the benchmark or solver after the deployed bundle has loaded.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import tempfile
import time
import zipfile
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
# How long this machine had been up when the notebook started, or None where /proc cannot tell: a Kaggle
# session can sit for long on its machine before the notebook runs.
try:
    MACHINE_UPTIME_S = float(Path("/proc/uptime").read_text().split()[0])
except (OSError, ValueError, IndexError):
    MACHINE_UPTIME_S = None
# Deploy-time emulation flag; Kaggle's real competition-rerun flag always wins.
RUN_AS_SUBMISSION = __TAAF_RUN_AS_SUBMISSION__  # noqa: F821
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = __TAAF_ENABLE_GPU__  # noqa: F821
# TOTAL runtime from notebook start at which a submission stops gracefully
# (read by _soft_end_time below). There is deliberately NO hard-exit
# fallback behind it: killing the kernel gets the notebook recorded as
# ERROR whatever the exit code — measured — and an ERROR run's submission
# is not scored, so a hard exit can only ever destroy a run that would
# have completed. If the graceful stop hangs, the platform kill ends the
# run with the same ERROR a hard exit would have produced.
SUBMISSION_STOP_RUNTIME_S = __TAAF_SUBMISSION_STOP_RUNTIME_S__  # noqa: F821

# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ.setdefault("MPLBACKEND", "Agg")

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")
machine_uptime = "unknown" if MACHINE_UPTIME_S is None else f"{MACHINE_UPTIME_S:.0f}s"
print(f"taaf.kaggle: machine up {machine_uptime} at notebook start")

In [ ]:
# ARC-AGI2's competition mount carries data only — there is no runtime to
# install from it, unlike ARC-AGI3's bundled wheels. Everything this run
# needs comes from the source dataset below.
print("taaf.kaggle: competition mount is data-only; no runtime install needed.")


In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES: list[str] = __TAAF_DATASET_SOURCES__  # noqa: F821
KERNEL_SOURCES: list[str] = __TAAF_KERNEL_SOURCES__  # noqa: F821
DATASET_BUNDLE_MARKER = __TAAF_DATASET_BUNDLE_MARKER__  # noqa: F821
DATASET_BUNDLE_ARCHIVE = __TAAF_DATASET_BUNDLE_ARCHIVE__  # noqa: F821
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", __TAAF_KAGGLE_WORKING_DIR__)).resolve()  # noqa: F821
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = __TAAF_SOFT_DEADLINE_BUFFER_S__  # noqa: F821
WORKING_DIR.mkdir(parents=True, exist_ok=True)


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    # Utility-script mount layouts vary per kernel: flat /kaggle/usr/lib/<slug>
    # or nested notebooks/<owner>/<slug>, and Kaggle may normalize hyphens to
    # underscores in the directory name (observed 2026-07: nested variant is
    # arc3_utils for the arc3-utils kernel). Probe every combination.
    candidates: list[Path] = []
    for name in dict.fromkeys([slug, slug.replace("-", "_")]):
        candidates.append(Path("/kaggle/usr/lib") / name)
        candidates.append(Path("/kaggle/usr/lib/notebooks") / owner / name)
    return candidates


def _first_existing(candidates: list[Path]) -> Path | None:
    # Prefer a candidate with actual content: an existing-but-empty mount dir
    # has been observed shadowing the populated variant.
    def _non_empty(path: Path) -> bool:
        try:
            return path.is_dir() and any(path.iterdir())
        except OSError:
            return False

    populated = next((candidate for candidate in candidates if _non_empty(candidate)), None)
    if populated is not None:
        return populated
    return next((candidate for candidate in candidates if candidate.exists()), None)


# The source dataset holds the bundle as one archive (a dataset of many files is slow to attach);
# it is unpacked to local disk, the bundle directory from then on.
def _unpack_bundle(archive: Path) -> Path:
    started = time.monotonic()
    bundle_dir = Path(tempfile.mkdtemp(prefix="taaf-bundle-"))
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(bundle_dir)
    print(f"taaf.kaggle: unpacked {archive} in {time.monotonic() - started:.1f}s", flush=True)
    return bundle_dir


# Locate the source dataset by its archive, or an unpacked bundle by its marker file, rather than a
# fixed mount path.
def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for archive in root.rglob(DATASET_BUNDLE_ARCHIVE):
                return _unpack_bundle(archive)
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent
    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
# Every layout is probed for every ref, whichever list it was declared in:
# Kaggle has attached one and the same source as a utility output and as
# a plain dataset at different times, so the declared kind is a preference
# for which path to report when NONE of them exists, not a promise about
# where the mount landed.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref) + _kernel_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref) + _dataset_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Audit attached datasets
import subprocess

subprocess.run(["ls", "/kaggle/input"], check=False)

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    def project_entries(project: Path) -> list[Path]:
        return [candidate for candidate in (project / "src", project) if candidate.is_dir()]

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        entries.extend(project_entries(repo))
        # A bundled uv workspace carries its members at their workspace-
        # relative paths; each member's source tree must be importable too.
        pyproject = repo / "pyproject.toml"
        if pyproject.is_file():
            import tomllib

            workspace = (
                tomllib.loads(pyproject.read_text(encoding="utf-8")).get("tool", {}).get("uv", {}).get("workspace")
                or {}
            )
            for pattern in workspace.get("members", []):
                for member in sorted(repo.glob(str(pattern))):
                    if member.is_dir() and (member / "pyproject.toml").is_file():
                        entries.extend(project_entries(member))
    return entries


# Environment handed to each setup/teardown command (paths + any keys it has persisted).
def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update(_load_setup_env())
    return env


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return
    commands = json.loads(path.read_text(encoding="utf-8"))
    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(str(command), shell=True, check=check, cwd=WORKING_DIR, env=env)
        if not check and result.returncode != 0:
            print(f"taaf.kaggle: {label} command exited with {result.returncode}", flush=True)
        # Re-read in case the command persisted new env keys.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")
    print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)", flush=True)

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Honour any PYTHONPATH a setup command exported through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

In [ ]:
# Both modes stop before their wall-clock line for the same graceful exit —
# cancel the solver, tear down, finish the remaining cells (measured, that
# teardown costs well under a minute). An ordinary run stops
# SOFT_DEADLINE_BUFFER_S before max_runtime_s; a submission stops AT
# SUBMISSION_STOP_RUNTIME_S, with no buffer of its own — the margin to
# Kaggle's platform kill already lives in that value (default:
# max_runtime_s − 240), and a second margin here would double-count it.
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    if run_as_submission:
        if SUBMISSION_STOP_RUNTIME_S <= 0:
            return None
        return start + timedelta(seconds=SUBMISSION_STOP_RUNTIME_S)
    if max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    return start + timedelta(seconds=budget - buffer)


# The earlier of the target's two startup deadlines: one counted from notebook start, one from machine
# boot, which also ends a session that sat for long on its machine before the notebook ran. The one from
# boot is the target's submission one in a submission, real or simulated.
def _startup_deadline(
    target, *, run_as_submission: bool, notebook_start: float, machine_uptime_s: float | None
) -> datetime | None:
    deadlines: list[datetime] = []
    after_start_s = getattr(target, "startup_deadline_s", None)
    if after_start_s is not None:
        deadlines.append(datetime.fromtimestamp(notebook_start) + timedelta(seconds=after_start_s))
    boot_field = "submission_startup_deadline_after_boot_s" if run_as_submission else "startup_deadline_after_boot_s"
    after_boot_s = getattr(target, boot_field, None)
    if after_boot_s is not None and machine_uptime_s is not None:
        deadlines.append(datetime.fromtimestamp(notebook_start - machine_uptime_s) + timedelta(seconds=after_boot_s))
    return min(deadlines, default=None)


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)
# The solver must be up by then, or the process exits: a hard exit, which a submission records as an error.
startup_deadline = _startup_deadline(
    target, run_as_submission=run_as_submission, notebook_start=NOTEBOOK_START_EPOCH, machine_uptime_s=MACHINE_UPTIME_S
)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Inline customization hook.
# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
# Example:
# bm.label = f'{bm.label}-debug'


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print(f"deploy.kaggle: startup_deadline        = {startup_deadline}")
    print("---")

    # Persist the launcher's git status for diagnostics.
    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # A scoring rerun is handed 240 hidden tasks we never configured, so
        # the game list is replaced by whatever the competition mounted, each
        # task offered exactly once. Nothing else about the run changes.
        import taaf.datasets
        import taaf.game

        specs = taaf.datasets.load(taaf.datasets.MOUNT_SET)
        bm.games = taaf.game.games_from_specs(specs)
        bm.n_passes = 1
        bm.game_weights = None
        print(f"deploy.kaggle: competition tasks       = {len(specs)}")

    # The submission is the deliverable and is written whatever the mode.
    bm.submission_path = WORKING_DIR / "submission.json"

    # Play the benchmark; teardown commands run even if the run raises.
    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
            startup_deadline=startup_deadline,
        )
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
# A non-submission run writes diagnostics.html; embed it in the saved notebook output.
from html import escape

diagnostics_html = WORKING_DIR / "diagnostics.html"
if not diagnostics_html.is_file():
    print("No diagnostics.html — minimal diagnostics (submission mode) suppresses it.")
else:
    try:
        from IPython.display import HTML, display
    except ImportError:
        # Local smoke tests run the cells outside IPython.
        print(f"diagnostics: {diagnostics_html}")
    else:
        # Isolate the full document in an iframe so its styles don't leak into the notebook.
        display(
            HTML(
                f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
                'width="100%" height="900" style="border:0"></iframe>'
            )
        )